# 01 - Importación

## Importar librerías

In [1]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime, timezone
import xml.etree.ElementTree as ET
import hashlib
import json
import requests
import pandas as pd

## Mejorar la visualización

In [2]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer rutas y visualización

In [3]:
RAIZ = Path.cwd().resolve()
for carpeta in [RAIZ, *RAIZ.parents]:
    if (carpeta / 'requirements.txt').is_file() and (carpeta / 'data').is_dir():
        RAIZ = carpeta
        break
else:
    raise FileNotFoundError('Abra el notebook desde el proyecto.')

RAW = RAIZ / 'data/raw'
REPORTS = RAIZ / 'data/reports'

## Descargar los archivos faltantes

In [4]:
def descargar(url, archivo):
    archivo.parent.mkdir(parents=True, exist_ok=True)
    if archivo.is_file():
        return archivo
    temporal = archivo.with_suffix(archivo.suffix + '.part')
    with requests.get(url, stream=True, timeout=(15, 90)) as respuesta:
        respuesta.raise_for_status()
        with temporal.open('wb') as salida:
            for bloque in respuesta.iter_content(1024 * 1024):
                salida.write(bloque)
        esperado = respuesta.headers.get('Content-Length')
        if esperado and not respuesta.headers.get('Content-Encoding'):
            assert temporal.stat().st_size == int(esperado), f'Descarga incompleta: {archivo.name}'
    assert temporal.stat().st_size > 0, f'Archivo vacío: {archivo.name}'
    if archivo.suffix == '.zip':
        with ZipFile(temporal) as paquete:
            assert paquete.testzip() is None, f'ZIP dañado: {archivo.name}'
    elif archivo.suffix == '.json':
        json.loads(temporal.read_text(encoding='utf-8-sig'))
    elif archivo.suffix == '.xml':
        ET.parse(temporal)
    elif archivo.suffix == '.html':
        assert '<html' in temporal.read_text(encoding='utf-8', errors='replace').lower()
    temporal.replace(archivo)
    return archivo

## Importar BACI

In [5]:
VERSION = '202601'
ANIOS = list(range(2017, 2025))
url_baci = f'https://www.cepii.fr/DATA_DOWNLOAD/baci/data/BACI_HS17_V{VERSION}.zip'
archivo_baci = descargar(url_baci, RAW / 'baci' / f'BACI_HS17_V{VERSION}.zip')

with ZipFile(archivo_baci) as paquete:
    miembros = [PurePosixPath(n).name for n in paquete.namelist()]
    assert all(f'BACI_HS17_Y{a}_V{VERSION}.csv' in miembros for a in ANIOS)
    for nombre in [f'country_codes_V{VERSION}.csv', f'product_codes_HS17_V{VERSION}.csv']:
        coincidencias = [n for n in paquete.namelist() if PurePosixPath(n).name == nombre]
        assert len(coincidencias) == 1, f'Catálogo ausente o repetido: {nombre}'
        destino = RAW / 'baci' / nombre
        if not destino.exists():
            destino.write_bytes(paquete.read(coincidencias[0]))

paises = pd.read_csv(RAW / 'baci' / f'country_codes_V{VERSION}.csv')
productos = pd.read_csv(RAW / 'baci' / f'product_codes_HS17_V{VERSION}.csv', dtype = {'code': str})
paises.head()

,country_code,country_name,country_iso2,country_iso3
0,4,Afghanistan,AF,AFG
1,8,Albania,AL,ALB
2,12,Algeria,DZ,DZA
3,16,American Samoa,AS,ASM
4,20,Andorra,AD,AND


In [6]:
productos.head()

,code,description
0,010121,"Horses: live, pure-bred breeding animals"
1,010129,"Horses: live, other than pure-bred breeding an..."
2,010130,Asses: live
3,010190,Mules and hinnies: live
4,010221,"Cattle: live, pure-bred breeding animals"


## Definir los indicadores WDI

In [7]:
indicadores = {'NY.GDP.MKTP.KD': 'gdp_real_usd'}
consultas = [('countries.json', 'https://api.worldbank.org/v2/country',
              {'format': 'json', 'per_page': 400})]
for indicador in indicadores:
    consultas.append((f'{indicador}_2017_2024.json',
        f'https://api.worldbank.org/v2/country/all/indicator/{indicador}',
        {'format': 'json', 'source': 2, 'date': '2017:2024', 'per_page': 20000}))

## Importar los datos WDI

In [8]:
carpeta_wdi = RAW / 'variables/world_bank'
carpeta_wdi.mkdir(parents=True, exist_ok=True)
for nombre, url, parametros in consultas:
    archivo = carpeta_wdi / nombre
    if not archivo.exists():
        registros, metadatos, pagina, paginas = [], [], 1, 1
        while pagina <= paginas:
            respuesta = requests.get(url, params = {**parametros, 'page': pagina}, timeout = (15, 60))
            respuesta.raise_for_status()
            meta, datos = respuesta.json()
            assert isinstance(datos, list), f'Respuesta WDI vacía: {nombre}'
            registros.extend(datos)
            metadatos.append(meta)
            paginas = int(meta['pages'])
            pagina += 1
        assert len(registros) == int(metadatos[0]['total'])
        contenido = {'url': url, 'parametros': parametros, 'registros': registros,
                     'metadatos_paginas': metadatos, 'fecha_descarga_utc': datetime.now(timezone.utc).isoformat()}
        temporal = archivo.with_suffix('.json.part')
        temporal.write_text(json.dumps(contenido, ensure_ascii=False), encoding='utf-8')
        temporal.replace(archivo)
    contenido = json.loads(archivo.read_text(encoding='utf-8'))
    assert contenido['url'] == url and contenido['parametros'] == parametros
    assert len(contenido['registros']) == int(contenido['metadatos_paginas'][0]['total'])

pd.DataFrame(contenido['registros']).head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'NY.GDP.MKTP.KD', 'value': 'GDP (consta...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,"1,102,001,360,919.78",,,0
1,"{'id': 'NY.GDP.MKTP.KD', 'value': 'GDP (consta...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,"1,072,111,623,592.75",,,0
2,"{'id': 'NY.GDP.MKTP.KD', 'value': 'GDP (consta...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,"1,051,694,225,642.19",,,0
3,"{'id': 'NY.GDP.MKTP.KD', 'value': 'GDP (consta...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,"1,014,482,704,419.61",,,0
4,"{'id': 'NY.GDP.MKTP.KD', 'value': 'GDP (consta...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2020,"971,236,989,040.35",,,0


## Importar las fuentes oficiales y complementarias

In [9]:
fuentes = [
    ('oficiales/Gravity_csv_V202211.zip', 'https://www.cepii.fr/DATA_DOWNLOAD/gravity/data/Gravity_csv_V202211.zip'),
    ('oficiales/wto_news_vars.js', 'https://www.wto.org/library/news/news_vars_e.js'),
    ('oficiales/oecd_lists.json', 'https://legalinstruments.oecd.org/api/lists?lang=en'),
    ('complementos_finales/un_m49.html', 'https://unstats.un.org/unsd/methodology/m49/overview/'),
    ('complementos_finales/taiwan_cuentas_nacionales.xml', 'https://ws.dgbas.gov.tw/001/Upload/461/relfile/11525/230514/na8101a1a.xml'),
    ('complementos_finales/cbs_topics.json', 'https://opendata.cbs.nl/ODataApi/OData/84789ENG/DataProperties'),
    ('complementos_finales/cbs_data.json', 'https://opendata.cbs.nl/ODataApi/OData/84789ENG/TypedDataSet'),
    ('complementos_finales/cbs_territorios.json', 'https://opendata.cbs.nl/ODataApi/OData/84789ENG/CarribeanNetherlands'),
]

for nombre, url in fuentes:
    descargar(url, RAW / 'variables' / nombre)

## Registrar los archivos importados

In [10]:
entradas = [(archivo_baci, url_baci)]
entradas += [(RAW / 'baci' / nombre, url_baci) for nombre in
             [f'country_codes_V{VERSION}.csv', f'product_codes_HS17_V{VERSION}.csv']]
entradas += [(carpeta_wdi / nombre, url) for nombre, url, _ in consultas]
entradas += [(RAW / 'variables' / nombre, url) for nombre, url in fuentes]
registros = []
for archivo, url in entradas:
    with archivo.open('rb') as entrada:
        huella = hashlib.file_digest(entrada, 'sha256').hexdigest()
    registros.append({'archivo': str(archivo.relative_to(RAIZ)), 'url': url,
                     'bytes': archivo.stat().st_size, 'sha256': huella})

REPORTS.mkdir(parents=True, exist_ok=True)
control = {'fecha_utc': datetime.now(timezone.utc).isoformat(), 'archivos': registros}
(REPORTS / '01_importacion.json').write_text(json.dumps(control, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(registros)[['archivo', 'bytes']]

,archivo,bytes
0,data\raw\baci\BACI_HS17_V202601.zip,794583540
1,data\raw\baci\country_codes_V202601.csv,5346
2,data\raw\baci\product_codes_HS17_V202601.csv,598831
3,data\raw\variables\world_bank\countries.json,192972
4,data\raw\variables\world_bank\NY.GDP.MKTP.KD_2...,770715
5,data\raw\variables\oficiales\Gravity_csv_V2022...,206707748
6,data\raw\variables\oficiales\wto_news_vars.js,273421
7,data\raw\variables\oficiales\oecd_lists.json,51133
8,data\raw\variables\complementos_finales\un_m49...,1721575
9,data\raw\variables\complementos_finales\taiwan...,361959
